## Imports

In [ ]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from pathlib import Path
from docx import Document as DocxDocument
from docx.table import Table
from docx.text.paragraph import Paragraph
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
import re

load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

s:\Capstone RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\saboo\AppData\Local\Temp\ipykernel_24436\2926881514.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4777.27it/s]


## Knowledge Base

In [3]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#",  "h1"),
        ("##", "h2")
    ],
    strip_headers=False
)

In [4]:
loader = DirectoryLoader(
    "Dataset/Knowledge Base Articles",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

docs = loader.load()


In [5]:
all_chunks = []

for doc in docs:
    chunks = splitter.split_text(doc.page_content)

    for chunk in chunks:
        chunk.metadata.update(doc.metadata) 

    all_chunks.extend(chunks)

In [ ]:
vectorstore = QdrantVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="knowledge_base_articles",
)

## FAQs

In [7]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#",  "h1"),
        ("##", "h2")
    ],
    strip_headers=False
)

In [8]:
loader = DirectoryLoader(
    "Dataset/FAQs",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

docs = loader.load()


In [9]:
all_chunks = []

for doc in docs:
    chunks = splitter.split_text(doc.page_content)

    for chunk in chunks:
        chunk.metadata.update(doc.metadata) 

    all_chunks.extend(chunks)

In [10]:
faq_chunks = []

for chunk in all_chunks:

    faq_entries = re.split(
        r'(?=\*\*Q:)',
        chunk.page_content
    )

    for entry in faq_entries:

        entry = entry.strip()

        if not entry.startswith("**Q:"):
            continue

        faq_chunks.append(
            Document(
                page_content=entry,
                metadata=chunk.metadata.copy()
            )
        )

In [11]:
faq_chunks.insert(0,all_chunks[0])

In [12]:
faq_chunks.append(all_chunks[-1])

In [ ]:
vectorstore = QdrantVectorStore.from_documents(
    documents=faq_chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="faqs",
)

## Product Documentation

In [14]:
def get_paragraph_font_size(paragraph):
    """
    Return first non-null font size found in paragraph.
    """
    for run in paragraph.runs:
        if run.font.size:
            return run.font.size.pt

    if paragraph.style and paragraph.style.font.size:
        return paragraph.style.font.size.pt

    return None


def iter_block_items(parent):
    """
    Iterate through paragraphs and tables
    in document order.
    """
    parent_elm = parent.element.body

    for child in parent_elm.iterchildren():

        if child.tag.endswith("}p"):
            yield Paragraph(child, parent)

        elif child.tag.endswith("}tbl"):
            yield Table(child, parent)


def table_to_text(table):
    """
    Convert a DOCX table into text.

    Single-column tables are treated as code blocks.
    Multi-column tables become markdown-like rows.
    """

    try:
        num_cols = len(table.columns)
    except:
        num_cols = None

    # ----------------------------------------
    # CODE BLOCK TABLE
    # ----------------------------------------
    if num_cols == 1:

        code_lines = []

        for row in table.rows:
            for cell in row.cells:

                text = cell.text.strip()

                if text:
                    code_lines.append(text)

        code = "\n".join(code_lines)

        return f"\n```bash\n{code}\n```\n"

    # ----------------------------------------
    # NORMAL TABLE
    # ----------------------------------------
    rows = []

    for row in table.rows:

        cells = [
            cell.text.strip().replace("\n", " ")
            for cell in row.cells
        ]

        rows.append(" | ".join(cells))

    return "\n".join(rows)


# --------------------------------------------------
# DOCX -> CHUNKS
# --------------------------------------------------

def load_docx_chunks(docx_path, heading_size=16):

    doc = DocxDocument(docx_path)

    document_header_lines = []

    chunks = []

    current_heading = None
    current_content = []

    found_first_heading = False

    # ----------------------------------------
    # PROCESS DOC IN ORDER
    # ----------------------------------------

    for block in iter_block_items(doc):

        # ====================================
        # PARAGRAPH
        # ====================================

        if isinstance(block, Paragraph):

            text = block.text.strip()

            if not text:
                continue

            font_size = get_paragraph_font_size(block)

            # --------------------------------
            # BEFORE FIRST HEADING
            # --------------------------------

            if not found_first_heading:

                if font_size == heading_size:

                    found_first_heading = True
                    current_heading = text

                else:
                    document_header_lines.append(text)

                continue

            # --------------------------------
            # NEW SECTION
            # --------------------------------

            if font_size == heading_size:

                if current_heading:

                    page_content = (
                        f"{current_heading}\n\n"
                        + "\n".join(current_content)
                    ).strip()

                    chunks.append(
                        Document(
                            page_content=page_content,
                            metadata={
                                "section": current_heading
                            }
                        )
                    )

                current_heading = text
                current_content = []

            else:
                current_content.append(text)

        # ====================================
        # TABLE
        # ====================================

        elif isinstance(block, Table):

            table_text = table_to_text(block)

            if table_text.strip():
                current_content.append(table_text)

    # ----------------------------------------
    # FINAL CHUNK
    # ----------------------------------------

    if current_heading:

        page_content = (
            f"{current_heading}\n\n"
            + "\n".join(current_content)
        ).strip()

        chunks.append(
            Document(
                page_content=page_content,
                metadata={
                    "section": current_heading
                }
            )
        )

    # ----------------------------------------
    # DOCUMENT METADATA
    # ----------------------------------------

    header_lines = [
        x.strip()
        for x in document_header_lines
        if x.strip()
    ]

    document_header = "\n".join(header_lines)

    product = header_lines[0] if len(header_lines) > 0 else ""
    document_title = header_lines[1] if len(header_lines) > 1 else ""
    document_info = header_lines[2] if len(header_lines) > 2 else ""

    # ----------------------------------------
    # ATTACH METADATA
    # ----------------------------------------

    for chunk in chunks:

        # Add document title to content
        if document_title:

            chunk.page_content = (
                f"{document_title}\n\n"
                f"{chunk.page_content}"
            )

        chunk.metadata.update({
            "source": str(docx_path),
            "filename": Path(docx_path).name,
            "product": product,
            "document_title": document_title,
            "document_info": document_info,
            "document_header": document_header,
        })

    return chunks


# --------------------------------------------------
# LOAD ALL DOCX FILES
# --------------------------------------------------

all_chunks = []

root_dir = Path(
    r"S:\Capstone RAG\Dataset\Product Documentaion"
)

for file in root_dir.rglob("*.docx"):

    try:

        docs = load_docx_chunks(file)

        print(
            f"{file.name}: {len(docs)} chunks"
        )

        all_chunks.extend(docs)

    except Exception as e:

        print(f"FAILED: {file}")
        print(e)

print(f"\nTotal chunks: {len(all_chunks)}")

docker_networking_reference.docx: 7 chunks
kubernetes_pod_scheduling_guide.docx: 7 chunks
python_stdlib_and_async.docx: 8 chunks
react_hooks_and_patterns.docx: 10 chunks

Total chunks: 32


In [ ]:
vectorstore = QdrantVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="product_documentation",
)